# 04 — Inference (Single Subject & Batch Prediction)
**Alzheimer's Disease Classification | KLE Technological University**

Loads saved model artifacts and predicts the diagnosis class (CN / MCI / AD) for new subjects.

### Required Inputs (per subject)
| Feature | Description | Example |
|---------|-------------|--------|
| `AGE` | Age in years | 78 |
| `PTEDUCAT` | Years of education | 14 |
| `MMSE` | Mini-Mental State Exam (0–30) | 22 |
| `ADAS13` | ADAS-Cog 13 score | 31 |
| `CDRSB` | CDR Sum of Boxes (0–18) | 6.0 |
| `Hippocampus` | Hippocampal volume mm³ | 4900 |
| `Entorhinal` | Entorhinal cortex volume mm³ | 1600 |

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
import joblib

MODELS_DIR = '/content/drive/MyDrive/AD1/lightgbm_final'

# Load artifacts
model_cog  = joblib.load(os.path.join(MODELS_DIR, 'lgb_model_cog.joblib'))
model_mri  = joblib.load(os.path.join(MODELS_DIR, 'lgb_model_mri.joblib'))
scaler_cog = joblib.load(os.path.join(MODELS_DIR, 'scaler_cog.joblib'))
scaler_mri = joblib.load(os.path.join(MODELS_DIR, 'scaler_mri.joblib'))
ens_params = joblib.load(os.path.join(MODELS_DIR, 'ensemble_params.joblib'))

w_cog, w_mri      = ens_params['w_cog'], ens_params['w_mri']
chosen_threshold  = ens_params['chosen_threshold']
cog_features      = ['AGE','AGE_sq','PTEDUCAT','MMSE','ADAS13','CDRSB','ADAS_gap']
mri_features      = ['Hippocampus','Entorhinal','H_E_ratio']
class_names       = ['CN','MCI','AD']

print('Models loaded successfully.')

In [ ]:
def predict_subject(input_dict, return_raw_probs=False):
    """
    Predict Alzheimer's disease class for a single subject.

    Parameters
    ----------
    input_dict : dict
        Keys: AGE, PTEDUCAT, MMSE, ADAS13, CDRSB, Hippocampus, Entorhinal
    return_raw_probs : bool
        If True, also return per-model probabilities.

    Returns
    -------
    dict with final_label, per-class ensemble probs, and AD probability.
    """
    required = ['AGE','PTEDUCAT','MMSE','ADAS13','CDRSB','Hippocampus','Entorhinal']
    missing  = [k for k in required if k not in input_dict]
    if missing:
        raise ValueError(f'Missing inputs: {missing}')

    AGE  = float(input_dict['AGE'])
    MMSE = float(input_dict['MMSE'])
    ADAS = float(input_dict['ADAS13'])
    Hipp = float(input_dict['Hippocampus'])
    Ent  = float(input_dict['Entorhinal'])

    # Derived features
    AGE_sq    = AGE ** 2
    ADAS_gap  = MMSE - ADAS
    H_E_ratio = Hipp / (Ent + 1e-9)

    Xc = pd.DataFrame([{
        'AGE': AGE, 'AGE_sq': AGE_sq,
        'PTEDUCAT': float(input_dict['PTEDUCAT']),
        'MMSE': MMSE, 'ADAS13': ADAS,
        'CDRSB': float(input_dict['CDRSB']),
        'ADAS_gap': ADAS_gap
    }])[cog_features]

    Xm = pd.DataFrame([{
        'Hippocampus': Hipp, 'Entorhinal': Ent, 'H_E_ratio': H_E_ratio
    }])[mri_features]

    Xc_s = pd.DataFrame(scaler_cog.transform(Xc), columns=cog_features)
    Xm_s = pd.DataFrame(scaler_mri.transform(Xm), columns=mri_features)

    pc   = model_cog.predict_proba(Xc_s)[0]
    pm   = model_mri.predict_proba(Xm_s)[0]
    pens = w_cog * pc + w_mri * pm

    ad_prob = float(pens[2])
    if ad_prob >= chosen_threshold:
        final_label = 'AD'
    else:
        final_label = class_names[int(np.argmax(pens[:2]))]

    out = {
        'final_label' : final_label,
        'probs'       : {class_names[i]: round(float(pens[i]),4) for i in range(3)},
        'ad_prob'     : round(ad_prob, 4)
    }
    if return_raw_probs:
        out['probs_cog'] = {class_names[i]: round(float(pc[i]),4) for i in range(3)}
        out['probs_mri'] = {class_names[i]: round(float(pm[i]),4) for i in range(3)}
    return out

In [ ]:
# --- Example: three representative subjects ---
samples = {
    'CN example' : {'AGE':70,'PTEDUCAT':16,'MMSE':29,'ADAS13':9, 'CDRSB':0.0,'Hippocampus':7200,'Entorhinal':3800},
    'MCI example': {'AGE':74,'PTEDUCAT':15,'MMSE':26,'ADAS13':18,'CDRSB':1.5,'Hippocampus':6100,'Entorhinal':2600},
    'AD example' : {'AGE':78,'PTEDUCAT':14,'MMSE':22,'ADAS13':31,'CDRSB':6.0,'Hippocampus':4900,'Entorhinal':1600}
}

for label, s in samples.items():
    res = predict_subject(s, return_raw_probs=True)
    print(f'\n=== {label} ===')
    print(f'  Final label     : {res["final_label"]}')
    print(f'  AD probability  : {res["ad_prob"]}')
    print(f'  Ensemble probs  : {res["probs"]}')
    print(f'  Cognitive probs : {res["probs_cog"]}')
    print(f'  MRI probs       : {res["probs_mri"]}')

In [ ]:
# --- Interactive single subject test ---
def safe_float(val, name):
    try:
        return float(val)
    except:
        raise ValueError(f'Invalid value for {name}: {val}')

def interactive_test():
    print('Enter the 7 required values (press Enter after each):')
    data = {
        'AGE'         : safe_float(input('AGE (e.g. 78): ').strip() or '0', 'AGE'),
        'PTEDUCAT'    : safe_float(input('PTEDUCAT (e.g. 14): ').strip() or '0', 'PTEDUCAT'),
        'MMSE'        : safe_float(input('MMSE 0-30 (e.g. 22): ').strip() or '0', 'MMSE'),
        'ADAS13'      : safe_float(input('ADAS13 (e.g. 31): ').strip() or '0', 'ADAS13'),
        'CDRSB'       : safe_float(input('CDRSB 0-18 (e.g. 6.0): ').strip() or '0', 'CDRSB'),
        'Hippocampus' : safe_float(input('Hippocampus mm³ (e.g. 4900): ').strip() or '0', 'Hippocampus'),
        'Entorhinal'  : safe_float(input('Entorhinal mm³ (e.g. 1600): ').strip() or '0', 'Entorhinal')
    }
    res = predict_subject(data, return_raw_probs=True)
    print('\n=== Prediction Result ===')
    print('Final label    :', res['final_label'])
    print('AD probability :', res['ad_prob'])
    print('Ensemble probs :', res['probs'])
    return res

# Uncomment to run:
# result = interactive_test()